# MAUSAM — Atmospheric Intelligence Platform
## Notebook 04: CPCB Air Quality Index & Health Impact Analysis

Analyzes ambient air quality monitoring telemetry from Continuous Ambient Air Quality Monitoring Stations (CAAQMS).
Computes Indian National Air Quality Index (CPCB NAQI) sub-indices using non-linear piecewise linear breakpoints across PM2.5, PM10, NO2, and SO2, synthesizes public health advisories, and correlates particulate surges with synoptic boundary layer wind stagnation.

In [1]:
import sys
import os
import json

sys.path.append(os.path.abspath('../python'))
from ingestion.cpcb_aqi_ingestion import CPCBAQIIngestion
from analytics.aqi_processor import AQIAnalyticsProcessor

# Telemetry recorded at Anand Vihar CAAQMS, New Delhi
station_telemetry = {
    "station_id": "DL-AV-01",
    "station_name": "Anand Vihar, New Delhi",
    "timestamp": "2026-09-09T08:00:00Z",
    "pm25": 168.4,
    "pm10": 285.0,
    "no2": 74.2,
    "so2": 18.5,
    "co": 1.4,
    "o3": 42.0
}

obs = CPCBAQIIngestion.parse_cpcb_station_record(station_telemetry)
advisory = AQIAnalyticsProcessor.generate_health_advisory(obs.aqi, obs.prominent_pollutant)

print(f"Computed NAQI: {obs.aqi} ({obs.category})")
print(f"Dominant Pollutant: {obs.prominent_pollutant}")
print(f"Sub-Indices: {obs.sub_indices}")
print("\nPublic Health Advisory:")
print(json.dumps(advisory, indent=2))

### 2. Multi-City Air Quality Index Comparison
Comparative analysis across Delhi, Mumbai, Kolkata, Chennai, and Bengaluru.

In [2]:
cities_telemetry = [
    {"name": "Delhi (Anand Vihar)", "pm25": 168.4, "pm10": 285.0, "no2": 74.2, "so2": 18.5},
    {"name": "Mumbai (Bandra)", "pm25": 38.5, "pm10": 68.0, "no2": 22.0, "so2": 12.0},
    {"name": "Kolkata (Victoria)", "pm25": 58.0, "pm10": 105.0, "no2": 36.0, "so2": 14.0},
    {"name": "Chennai (Alandur)", "pm25": 24.0, "pm10": 48.0, "no2": 16.0, "so2": 8.0},
    {"name": "Bengaluru (BTM)", "pm25": 28.5, "pm10": 52.0, "no2": 19.5, "so2": 7.5}
]

city_results = []
for c in cities_telemetry:
    res = CPCBAQIIngestion.parse_cpcb_station_record({
        "station_id": c["name"],
        "station_name": c["name"],
        "timestamp": "2026-09-09T08:00:00Z",
        **c
    })
    city_results.append({
        "city": c["name"],
        "aqi": res.aqi,
        "category": res.category,
        "prominent": res.prominent_pollutant
    })

print(json.dumps(city_results, indent=2))